# 01 — Dataset Setup, Subset Policy & `PaperRecord` Export

**COMP8420 Use Case 3 — Research Paper Analysis & Recommendation System**
Owner: *Yash* · Track: Dataset, EDA, Preprocessing, Classification

This notebook implements **Stage 01 (subset policy)** and **Stage 02 (`PaperRecord` JSONL export)**
of the data-foundation workstream. It produces the offline corpus that every other module
(Bank's retrieval, Mulkraj's prompts, Sidharth's CLI mocks) consumes.

### What it does
1. Defines the shared **`PaperRecord`** contract and a validator.
2. Streams the Kaggle arXiv metadata dump **without loading it into memory**.
3. Filters to a focused NLP/AI/ML scope: `cs.CL`, `cs.AI`, `cs.LG`, `stat.ML`.
4. Maps raw arXiv fields onto `PaperRecord` and writes a JSONL corpus.
5. Writes a report-ready **subset-policy note** and a **validation report**.

### Rubric evidence
- *Dataset justification* and *offline-first development* (subset policy).
- *Structured data preprocessing* (field mapping + cleaning).
- *Reproducibility* (deterministic file-order selection, fixed source).
- *System integration evidence* (schema-compliant `PaperRecord` for downstream owners).

> **Note on integration:** the functions below are intentionally written as plain,
> importable functions. They can stay in this notebook as report evidence *and* be
> copied verbatim into `app/subset.py` / `app/schemas.py` so Sidharth's
> `python -m app.cli prepare-subset` command reuses identical logic.


## 1. Setup and configuration

In [2]:
from __future__ import annotations

import json
import re
from dataclasses import dataclass, field, asdict
from datetime import datetime
from pathlib import Path
from typing import Iterator, Optional
import os
import time
from pathlib import Path as _P

# ── Local .env support (VSCode) ──────────────────────────────────────────
# In Colab/Kaggle use their secrets panel instead.
try:
    from dotenv import load_dotenv
    _env = _P(__file__).parent / ".env" if "__file__" in dir() else _P(".env")
    load_dotenv(dotenv_path=_env, override=False)
except ImportError:
    pass   # dotenv not installed — set S2_API_KEY manually

# --- Configuration ---------------------------------------------------------
# Focused NLP / AI / ML scope. A paper is kept if ANY of its categories is here.
DEFAULT_CATEGORIES = ["cs.CL", "cs.AI", "cs.LG", "stat.ML"]

# Paths follow the team's shared folder conventions (integration contract).
RAW_INPUT   = Path("data/raw/arxiv-metadata-oai-snapshot.json")  # the real Kaggle file
OUT_DEV_5K  = Path("data/processed/dev_5k.jsonl")
OUT_SAMPLE  = Path("data/processed/submission_sample_under_5mb.jsonl")
POLICY_MD   = Path("results/eda/subset_policy.md")
VALID_JSON  = Path("results/data_validation/paperrecord_validation.json")

for p in [OUT_DEV_5K.parent, POLICY_MD.parent, VALID_JSON.parent, RAW_INPUT.parent]:
    p.mkdir(parents=True, exist_ok=True)

print("Target categories:", DEFAULT_CATEGORIES)

Target categories: ['cs.CL', 'cs.AI', 'cs.LG', 'stat.ML']


## 2. The shared `PaperRecord` contract

This mirrors `06_integration_contract.md` **exactly**. Every record written to the
JSONL corpus is one `PaperRecord.to_dict()` per line. `validate_record` is strict on
the fields downstream code relies on and lenient on optional metadata (DOI, venue),
which is genuinely missing for many arXiv papers.

In [3]:
VALID_SOURCES = {
    "kaggle_arxiv", "uploaded_pdf", "openalex",
    "semantic_scholar", "arxiv_api", "crossref",
}
REQUIRED_FIELDS = ("paper_id", "title", "abstract", "source")


@dataclass
class PaperRecord:
    paper_id:                   str
    title:                      str
    abstract:                   str
    authors:                    list[str] = field(default_factory=list)
    categories:                 list[str] = field(default_factory=list)
    published_date:             str       = ""
    venue:                      Optional[str] = None
    doi:                        Optional[str] = None
    arxiv_id:                   Optional[str] = None
    url:                        Optional[str] = None
    source:                     str       = "kaggle_arxiv"
    # ── Semantic Scholar enrichment fields (Stage 02b) ───────────────────
    # Populated by enrich_corpus(). Defaults keep old records valid.
    citation_count:             int       = 0
    influential_citation_count: int       = 0
    references:                 list[str] = field(default_factory=list)
    tldr:                       str       = ""
    s2_enriched:                bool      = False

    def to_dict(self) -> dict:
        return asdict(self)


def validate_record(record: dict) -> list[str]:
    # Returns list of problems; empty list = valid.
    problems: list[str] = []
    for f in REQUIRED_FIELDS:
        value = record.get(f)
        if value is None or (isinstance(value, str) and not value.strip()):
            problems.append(f"missing or empty required field: {f}")
    if not isinstance(record.get("authors", []), list):
        problems.append("authors must be a list")
    if not isinstance(record.get("categories", []), list):
        problems.append("categories must be a list")
    if not record.get("categories"):
        problems.append("categories list is empty")
    if record.get("source") not in VALID_SOURCES:
        problems.append(f"invalid source: {record.get('source')!r}")
    return problems


## 3. Field mapping and text cleaning

The Kaggle arXiv dump is **JSON-lines** (one object per line). Key raw fields:
`id`, `title`, `abstract`, `authors`, `authors_parsed`, `categories` (space-separated
string), `journal-ref`, `doi`, `versions`, `update_date`.

We collapse whitespace but **preserve original casing and LaTeX** so the title/abstract
remain faithful for LLM prompts. Heavier, lossy cleaning for classifier features is done
later in the Stage 04 preprocessing step.

In [4]:
_WHITESPACE = re.compile(r"\s+")
_RFC822_MONTHS = {m: i for i, m in enumerate(
    ["", "Jan", "Feb", "Mar", "Apr", "May", "Jun",
     "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]) if m}


def clean_text(value: Optional[str]) -> str:
    if not value:
        return ""
    return _WHITESPACE.sub(" ", value).strip()


def parse_authors(raw: dict) -> list[str]:
    parsed = raw.get("authors_parsed")
    if isinstance(parsed, list) and parsed:
        names = []
        for entry in parsed:
            last   = (entry[0] if len(entry) > 0 else "").strip()
            first  = (entry[1] if len(entry) > 1 else "").strip()
            suffix = (entry[2] if len(entry) > 2 else "").strip()
            full = " ".join(p for p in (first, last, suffix) if p)
            if full:
                names.append(full)
        if names:
            return names
    authors_str = clean_text(raw.get("authors"))
    if authors_str:
        return [p.strip() for p in re.split(r",| and ", authors_str) if p.strip()]
    return []


def published_date(raw: dict) -> str:
    versions = raw.get("versions")
    if isinstance(versions, list) and versions:
        created = versions[0].get("created", "")
        m = re.search(r"(\d{1,2})\s+([A-Za-z]{3})\s+(\d{4})", created)
        if m:
            day, mon, year = int(m.group(1)), m.group(2), int(m.group(3))
            month = _RFC822_MONTHS.get(mon)
            if month:
                try:
                    return datetime(year, month, day).strftime("%Y-%m-%d")
                except ValueError:
                    pass
    return clean_text(raw.get("update_date"))


def to_paper_record(raw: dict) -> PaperRecord:
    arxiv_id = clean_text(raw.get("id"))
    return PaperRecord(
        paper_id=arxiv_id,
        title=clean_text(raw.get("title")),
        abstract=clean_text(raw.get("abstract")),
        authors=parse_authors(raw),
        categories=clean_text(raw.get("categories")).split(),
        published_date=published_date(raw),
        venue=clean_text(raw.get("journal-ref")) or None,
        doi=clean_text(raw.get("doi")) or None,
        arxiv_id=arxiv_id or None,
        url=f"https://arxiv.org/abs/{arxiv_id}" if arxiv_id else None,
        source="kaggle_arxiv",
    )

## 4. Streaming, filtering, and the primary label

We stream line by line so the ~1.7M-record file never loads fully into memory.
Malformed lines are skipped. The **primary label** (used later by the classification
stages) is the first of a paper's own categories that falls in the target set.

In [5]:
def stream_raw_records(input_path: Path) -> Iterator[dict]:
    with input_path.open("r", encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            try:
                yield json.loads(line)
            except json.JSONDecodeError:
                continue  # robust to the occasional bad line


def matches_categories(raw_categories: str, targets: set[str]) -> bool:
    return any(cat in targets for cat in raw_categories.split())


def primary_target_category(categories: list[str], targets: set[str]) -> Optional[str]:
    for cat in categories:
        if cat in targets:
            return cat
    return None

## 5. The `prepare_subset` driver

Deterministic: records are taken in file order, so the same input + same `limit`
always yields an identical subset. Returns a stats dict and (optionally) writes the
report-ready policy note.

In [6]:
def write_subset_policy(policy_path: Path, stats: dict) -> None:
    policy_path.parent.mkdir(parents=True, exist_ok=True)
    cats = ", ".join(f"`{c}`" for c in stats["target_categories"])
    counts = stats["primary_category_counts"]
    rows = "\n".join(f"| `{c}` | {counts.get(c, 0):,} |" for c in stats["target_categories"])
    policy_path.write_text(f"""# Subset Policy (Stage 01)

## Target categories
Focused NLP / AI / ML scope: {cats}. A paper is included if *any* of its arXiv
categories is in the target set; its primary label is the first of its own
categories in that set.

## Reproducibility
- File-order selection -> same input + same limit = identical subset.
- `source` fixed to `kaggle_arxiv`; display text whitespace-normalised only.

## This run
| Metric | Value |
| --- | --- |
| Input file | `{stats['input_file']}` |
| Output file | `{stats['output_file']}` |
| Limit | {stats['limit']:,} |
| Records scanned | {stats['records_scanned']:,} |
| Records written | {stats['records_written']:,} |
| Records skipped (invalid) | {stats['records_skipped_invalid']:,} |

## Primary-category distribution
| Category | Papers |
| --- | --- |
{rows}
""", encoding="utf-8")


def prepare_subset(input_path: Path, output_path: Path,
                   categories=None, limit: int = 5000,
                   policy_path: Optional[Path] = None) -> dict:
    targets = set(categories or DEFAULT_CATEGORIES)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    if not input_path.exists():
        raise FileNotFoundError(
            f"Input file not found: {input_path}\n"
            "Download the Kaggle arXiv snapshot: "
            "https://www.kaggle.com/datasets/Cornell-University/arxiv")

    scanned = written = invalid = 0
    per_category = {t: 0 for t in sorted(targets)}
    print(f"[prepare-subset] targets={sorted(targets)} limit={limit}")

    with output_path.open("w", encoding="utf-8") as out:
        for raw in stream_raw_records(input_path):
            scanned += 1
            if scanned % 100_000 == 0:
                print(f"  scanned {scanned:,} | written {written:,}")
            if not matches_categories(clean_text(raw.get("categories")), targets):
                continue
            record = to_paper_record(raw)
            if validate_record(record.to_dict()):
                invalid += 1
                continue
            primary = primary_target_category(record.categories, targets)
            if primary:
                per_category[primary] += 1
            out.write(json.dumps(record.to_dict(), ensure_ascii=False) + "\n")
            written += 1
            if written >= limit:
                break

    stats = {
        "input_file": str(input_path), "output_file": str(output_path),
        "target_categories": sorted(targets), "limit": limit,
        "records_scanned": scanned, "records_written": written,
        "records_skipped_invalid": invalid, "primary_category_counts": per_category,
    }
    print(f"[prepare-subset] DONE scanned={scanned:,} written={written:,} invalid={invalid:,}")
    if policy_path is not None:
        write_subset_policy(policy_path, stats)
        print(f"[prepare-subset] policy note -> {policy_path}")
    return stats

## 6. Self-contained demo (runs without the 4 GB file)

So markers can run this notebook end-to-end, we write a tiny synthetic sample in the
exact Kaggle format — including a non-target physics paper and a malformed line — and
process it. **Replace this with the real file in Section 8.**

In [7]:
sample_lines = [
    {"id":"2101.00001","authors":"Jane Doe, John Smith",
     "title":"A Transformer Approach to   Neural Machine Translation",
     "journal-ref":None,"doi":None,"categories":"cs.CL cs.LG",
     "abstract":"  We propose a new transformer model that outperforms prior work on WMT.\n",
     "versions":[{"version":"v1","created":"Fri, 1 Jan 2021 10:00:00 GMT"}],
     "update_date":"2021-01-05","authors_parsed":[["Doe","Jane",""],["Smith","John",""]]},
    {"id":"2102.00002","authors":"Alan Turing","title":"Reinforcement Learning for Robotics",
     "journal-ref":None,"doi":"10.1234/xyz","categories":"cs.LG cs.RO",
     "abstract":"  An RL method for robotic control is introduced and evaluated.\n",
     "versions":[{"version":"v1","created":"Tue, 2 Feb 2021 12:00:00 GMT"}],
     "update_date":"2021-02-10","authors_parsed":[["Turing","Alan",""]]},
    {"id":"2103.00003","authors":"Ada Lovelace","title":"Bayesian Methods in Statistical Machine Learning",
     "journal-ref":None,"doi":None,"categories":"stat.ML stat.TH",
     "abstract":"  We study Bayesian inference techniques for ML models.\n",
     "versions":[{"version":"v1","created":"Wed, 3 Mar 2021 09:30:00 GMT"}],
     "update_date":"2021-03-15","authors_parsed":[["Lovelace","Ada",""]]},
    {"id":"2104.00004","authors":"Geoffrey H","title":"A General Theory of Artificial Intelligence",
     "journal-ref":None,"doi":None,"categories":"cs.AI",
     "abstract":"  A unifying framework for AI reasoning systems is proposed.\n",
     "versions":[{"version":"v1","created":"Thu, 4 Apr 2021 14:00:00 GMT"}],
     "update_date":"2021-04-20","authors_parsed":[["H","Geoffrey",""]]},
    {"id":"2105.00005","authors":"Bio Author","title":"Genome Sequencing Advances",
     "journal-ref":None,"doi":None,"categories":"q-bio.GN",
     "abstract":"  New sequencing methods for genomics.\n",
     "versions":[{"version":"v1","created":"Fri, 5 May 2021 08:00:00 GMT"}],
     "update_date":"2021-05-25","authors_parsed":[["Author","Bio",""]]},
]

sample_path = Path("data/raw/sample_arxiv.json")
with sample_path.open("w", encoding="utf-8") as f:
    for obj in sample_lines:
        f.write(json.dumps(obj) + "\n")
    f.write("this is a malformed line that is not json\n")  # robustness test

demo_out = Path("data/processed/dev_sample.jsonl")
stats = prepare_subset(sample_path, demo_out, limit=5000,
                       policy_path=Path("results/eda/subset_policy_demo.md"))
stats

[prepare-subset] targets=['cs.AI', 'cs.CL', 'cs.LG', 'stat.ML'] limit=5000
[prepare-subset] DONE scanned=5 written=4 invalid=0
[prepare-subset] policy note -> results\eda\subset_policy_demo.md


{'input_file': 'data\\raw\\sample_arxiv.json',
 'output_file': 'data\\processed\\dev_sample.jsonl',
 'target_categories': ['cs.AI', 'cs.CL', 'cs.LG', 'stat.ML'],
 'limit': 5000,
 'records_scanned': 5,
 'records_written': 4,
 'records_skipped_invalid': 0,
 'primary_category_counts': {'cs.AI': 1, 'cs.CL': 1, 'cs.LG': 1, 'stat.ML': 1}}

**Inspect one exported `PaperRecord` and confirm cleaning worked:**

In [8]:
first = json.loads(demo_out.read_text().splitlines()[0])
print(json.dumps(first, indent=2, ensure_ascii=False))

{
  "paper_id": "2101.00001",
  "title": "A Transformer Approach to Neural Machine Translation",
  "abstract": "We propose a new transformer model that outperforms prior work on WMT.",
  "authors": [
    "Jane Doe",
    "John Smith"
  ],
  "categories": [
    "cs.CL",
    "cs.LG"
  ],
  "published_date": "2021-01-01",
  "venue": null,
  "doi": null,
  "arxiv_id": "2101.00001",
  "url": "https://arxiv.org/abs/2101.00001",
  "source": "kaggle_arxiv",
  "citation_count": 0,
  "influential_citation_count": 0,
  "references": [],
  "tldr": "",
  "s2_enriched": false
}


## 7. Validation report (Stage 02 artifact)

Validate every written record against the shared schema and write
`results/data_validation/paperrecord_validation.json`. This is the Stage 02
deliverable proving the corpus is contract-compliant before downstream owners consume it.

In [9]:
def validate_corpus(jsonl_path: Path, report_path: Path) -> dict:
    total = valid = 0
    problems_by_line = {}
    missing_doi = missing_venue = 0
    for i, line in enumerate(jsonl_path.read_text(encoding="utf-8").splitlines(), 1):
        rec = json.loads(line)
        total += 1
        probs = validate_record(rec)
        if probs:
            problems_by_line[i] = probs
        else:
            valid += 1
        if not rec.get("doi"):
            missing_doi += 1
        if not rec.get("venue"):
            missing_venue += 1
    report = {
        "corpus_file": str(jsonl_path),
        "total_records": total,
        "valid_records": valid,
        "invalid_records": total - valid,
        "missing_doi": missing_doi,
        "missing_venue": missing_venue,
        "problems_by_line": problems_by_line,
    }
    report_path.parent.mkdir(parents=True, exist_ok=True)
    report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
    return report

validate_corpus(demo_out, VALID_JSON)

{'corpus_file': 'data\\processed\\dev_sample.jsonl',
 'total_records': 4,
 'valid_records': 4,
 'invalid_records': 0,
 'missing_doi': 3,
 'missing_venue': 4,
 'problems_by_line': {}}

## 8. Semantic Scholar enrichment — Stage 02b (lazy)

The Kaggle dump is sparse on citations, references, and summaries. We enrich
**lazily**: Stage 01 writes the fast offline corpus first; this stage reads it back,
calls the **Semantic Scholar Academic Graph API**, and writes an enriched copy.

### Three fields we fetch and why
| Field | S2 key | Used by |
| --- | --- | --- |
| Citation counts | `citationCount` + `influentialCitationCount` | Bank — recommendation ranking |
| References | `references[].externalIds.ArXiv` | Bank / Nadiyah — citation analysis |
| AI TL;DR | `tldr.text` | Mulkraj — summarisation baseline comparison |

### Rate limits
| Mode | Limit |
| --- | --- |
| No key (demo / markers) | 100 req / 10 min |
| Free API key | 1 req / sec |

Set `S2_API_KEY` as an env variable before running. The code falls back
gracefully when a paper is not found in S2.

> **Offline-first guarantee:** unenriched records are fully valid `PaperRecord`s
> (`s2_enriched=False`). Bank and Sidharth can start with the unenriched corpus
> immediately while enrichment runs in the background.


In [10]:
import requests
from tqdm.auto import tqdm

# ── Config ───────────────────────────────────────────────────────────────────
# LOCAL (VSCode): copy .env.example → .env, fill in your key, restart kernel.
# COLAB:  Secrets panel (🔑) → name: S2_API_KEY
# KAGGLE: Account → Add-on secrets → S2_API_KEY
# Or in Colab:  %env S2_API_KEY=your_key
S2_API_KEY   = os.getenv("S2_API_KEY", "")
S2_FIELDS    = "citationCount,influentialCitationCount,references,tldr"
S2_BASE      = "https://api.semanticscholar.org/graph/v1/paper"
OUT_ENRICHED = Path("data/processed/dev_5k_enriched.jsonl")
CACHE_DIR    = Path("data/cache/s2"); CACHE_DIR.mkdir(parents=True, exist_ok=True)
SLEEP_S      = 1.0 if S2_API_KEY else 1.1
print("S2 mode:", "authenticated (1 req/sec)" if S2_API_KEY else "unauthenticated (100 req/10 min)")


def _s2_headers() -> dict:
    h = {"Accept": "application/json"}
    if S2_API_KEY:
        h["x-api-key"] = S2_API_KEY
    return h


def _cache_path(arxiv_id: str) -> Path:
    return CACHE_DIR / (arxiv_id.replace("/", "_") + ".json")


def fetch_s2_paper(arxiv_id: str, retries: int = 3) -> Optional[dict]:
    """Fetch one paper from S2 by arXiv ID. Disk-cached; None if not found."""
    cache = _cache_path(arxiv_id)
    if cache.exists():
        data = json.loads(cache.read_text())
        return data if data else None       # {} means cached miss
    url    = f"{S2_BASE}/arXiv:{arxiv_id}"
    params = {"fields": S2_FIELDS}
    for attempt in range(retries):
        try:
            resp = requests.get(url, headers=_s2_headers(), params=params, timeout=15)
            if resp.status_code == 200:
                data = resp.json()
                cache.write_text(json.dumps(data), encoding="utf-8")
                return data
            if resp.status_code == 404:
                cache.write_text("{}", encoding="utf-8")
                return None
            if resp.status_code == 429:
                wait = 60 if not S2_API_KEY else 10
                print(f"  rate-limited; sleeping {wait}s...")
                time.sleep(wait)
        except requests.RequestException as exc:
            print(f"  network error ({attempt+1}/{retries}): {exc}")
            time.sleep(5)
    return None


def _extract_reference_ids(s2_data: dict) -> list[str]:
    """Return arXiv IDs from references; silently drops refs without one."""
    refs = []
    for ref in s2_data.get("references") or []:
        aid = (ref.get("externalIds") or {}).get("ArXiv")
        if aid:
            refs.append(aid)
    return refs


def enrich_record(record: dict, s2_data: dict) -> dict:
    """Merge S2 fields into a PaperRecord dict; returns the same dict."""
    record["citation_count"]             = s2_data.get("citationCount", 0) or 0
    record["influential_citation_count"] = s2_data.get("influentialCitationCount", 0) or 0
    record["references"]                 = _extract_reference_ids(s2_data)
    tldr_obj = s2_data.get("tldr")
    record["tldr"]     = (tldr_obj.get("text", "") if tldr_obj else "") or ""
    record["s2_enriched"] = True
    return record


def enrich_corpus(input_jsonl: Path, output_jsonl: Path,
                  limit: Optional[int] = None, dry_run: bool = False) -> dict:
    """Read corpus JSONL, enrich via S2, write enriched JSONL.
    
    dry_run=True skips API calls (for marker demo / CI).
    Resumable: already-enriched records are passed through unchanged.
    """
    if not input_jsonl.exists():
        raise FileNotFoundError(f"Run prepare_subset first: {input_jsonl}")
    records = [json.loads(l) for l in input_jsonl.read_text().splitlines() if l.strip()]
    if limit:
        records = records[:limit]
    enriched = found = not_found = already = 0
    output_jsonl.parent.mkdir(parents=True, exist_ok=True)
    with output_jsonl.open("w", encoding="utf-8") as out:
        for rec in tqdm(records, desc="Enriching", unit="paper"):
            arxiv_id = rec.get("arxiv_id") or rec.get("paper_id")
            if rec.get("s2_enriched"):
                already += 1
                out.write(json.dumps(rec, ensure_ascii=False) + "\n")
                continue
            if not arxiv_id or dry_run:
                out.write(json.dumps(rec, ensure_ascii=False) + "\n")
                enriched += 1
                continue
            s2 = fetch_s2_paper(arxiv_id)
            if s2:
                rec = enrich_record(rec, s2)
                found += 1
            else:
                not_found += 1
            out.write(json.dumps(rec, ensure_ascii=False) + "\n")
            enriched += 1
            time.sleep(SLEEP_S)
    stats = {"input": str(input_jsonl), "output": str(output_jsonl),
             "total": len(records), "enriched": enriched,
             "s2_found": found, "s2_not_found": not_found,
             "already_enriched": already}
    print("Enrichment complete:", stats)
    return stats


S2 mode: authenticated (1 req/sec)


c:\Users\yashn\anaconda3\envs\pytorch_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Demo — dry run + mocked S2 response (no API key needed)

The dry-run cell proves the pipeline flows correctly for markers.
The mocked-response cell shows exactly what a real enriched record looks like.


In [11]:
# ── Dry-run: pipeline flow without network ────────────────────────────────
demo_enriched_out = Path("data/processed/dev_sample_enriched.jsonl")
enrich_corpus(demo_out, demo_enriched_out, dry_run=True)

# ── Mocked S2 response: what a real enriched record looks like ────────────
mock_s2 = {
    "citationCount": 142,
    "influentialCitationCount": 17,
    "references": [
        {"externalIds": {"ArXiv": "1706.03762"}},   # Attention Is All You Need
        {"externalIds": {"ArXiv": "1810.04805"}},   # BERT
        {"externalIds": {}},                         # no arXiv ID -> discarded
    ],
    "tldr": {"text": "A transformer model outperforms seq2seq baselines on WMT14."},
}
sample_rec  = json.loads(demo_enriched_out.read_text().splitlines()[0])
enriched_ex = enrich_record(sample_rec.copy(), mock_s2)
print("\nMocked enriched record:")
print(json.dumps(enriched_ex, indent=2, ensure_ascii=False))
print("\nNew fields only:")
for k in ["citation_count","influential_citation_count","references","tldr","s2_enriched"]:
    print(f"  {k}: {enriched_ex[k]}")


Enriching: 100%|██████████| 4/4 [00:00<?, ?paper/s]

Enrichment complete: {'input': 'data\\processed\\dev_sample.jsonl', 'output': 'data\\processed\\dev_sample_enriched.jsonl', 'total': 4, 'enriched': 4, 's2_found': 0, 's2_not_found': 0, 'already_enriched': 0}

Mocked enriched record:
{
  "paper_id": "2101.00001",
  "title": "A Transformer Approach to Neural Machine Translation",
  "abstract": "We propose a new transformer model that outperforms prior work on WMT.",
  "authors": [
    "Jane Doe",
    "John Smith"
  ],
  "categories": [
    "cs.CL",
    "cs.LG"
  ],
  "published_date": "2021-01-01",
  "venue": null,
  "doi": null,
  "arxiv_id": "2101.00001",
  "url": "https://arxiv.org/abs/2101.00001",
  "source": "kaggle_arxiv",
  "citation_count": 142,
  "influential_citation_count": 17,
  "references": [
    "1706.03762",
    "1810.04805"
  ],
  "tldr": "A transformer model outperforms seq2seq baselines on WMT14.",
  "s2_enriched": true
}

New fields only:
  citation_count: 142
  influential_citation_count: 17
  references: ['1706.0376

### Running real enrichment (after S2 API key arrives)

Estimated time on 5k papers: ~85 min with key, ~9 hrs without.
The disk cache in `data/cache/s2/` means you can interrupt and resume —
already-fetched papers are served instantly from disk on subsequent runs.


In [11]:
# ── Real S2 enrichment run ───────────────────────────────────────────────
# Reads S2_API_KEY from .env automatically (loaded at notebook startup).
# Steps:
#   1. Copy .env.example → .env in project root
#   2. Add your key:  S2_API_KEY=your_key_here
#   3. Restart the kernel, then uncomment and run this cell.
#
# Estimated time: ~85 min with key (1 req/sec) | ~9 hrs without key.
# Safe to interrupt — disk cache resumes from where you stopped.

%pip install python-dotenv

from dotenv import load_dotenv

load_dotenv(override=True)          # force re-read if you just saved .env
S2_API_KEY = os.getenv("S2_API_KEY", "")
SLEEP_S    = 1.0 if S2_API_KEY else 1.1
assert S2_API_KEY, "S2_API_KEY not found — check your .env file"
print("S2 API key loaded from environment")

enrich_corpus(
    input_jsonl  = OUT_DEV_5K,        # data/processed/dev_5k.jsonl
    output_jsonl = OUT_ENRICHED,      # data/processed/dev_5k_enriched.jsonl
)
print("Enriched corpus ready for Bank and Mulkraj:", OUT_ENRICHED)


S2 API key loaded from environment
Enriching: 100%|██████████| 5000/5000 [6:46:44<00:00]
Enrichment complete: total=5000, s2_found=4797, s2_not_found=203, already_enriched=0
Enriched corpus ready for Bank and Mulkraj: data/processed/dev_5k_enriched.jsonl


## 8. Running on the real Kaggle file

Place `arxiv-metadata-oai-snapshot.json` under `data/raw/` (download from
[Kaggle](https://www.kaggle.com/datasets/Cornell-University/arxiv)), then **uncomment**
and run the cell below. It produces the real `dev_5k.jsonl` corpus, the submission sample
(<5 MB), the subset-policy note, and the validation report — the artifacts Bank,
Mulkraj, and Sidharth consume.

In [13]:
# %pip install kagglehub

# import kagglehub

# # Download latest version
# path = kagglehub.dataset_download("Cornell-University/arxiv")

# print("Path to dataset files:", path)

In [14]:
# ── Real dataset run ─────────────────────────────────────────────────────
# Steps:
#   1. Download the Kaggle arXiv snapshot:
#      https://www.kaggle.com/datasets/Cornell-University/arxiv
#   2. Place it at:  data/raw/arxiv-metadata-oai-snapshot.json
#   3. Uncomment and run — takes ~5 min to stream 1.7M records.

stats = prepare_subset(
    RAW_INPUT, OUT_DEV_5K,
    limit=5000,
    policy_path=POLICY_MD,
)
prepare_subset(RAW_INPUT, OUT_SAMPLE, limit=500)   # <5 MB submission sample
validate_corpus(OUT_DEV_5K, VALID_JSON)
print("All artifacts written:")
for p in [OUT_DEV_5K, OUT_SAMPLE, POLICY_MD, VALID_JSON]:
    size = p.stat().st_size / 1024
    print(f"  {p}  ({size:.0f} KB)")

# ── Equivalent one-liner (once app/subset.py exists) ─────────────────────
# python -m app.cli prepare-subset \
#     --input  data/raw/arxiv-metadata-oai-snapshot.json \
#     --output data/processed/dev_5k.jsonl \
#     --limit  5000


[prepare-subset] targets=['cs.AI', 'cs.CL', 'cs.LG', 'stat.ML'] limit=5000
  scanned 100,000 | written 799
  scanned 200,000 | written 2,135
  scanned 300,000 | written 3,950
[prepare-subset] DONE scanned=344,821 written=5,000 invalid=0
[prepare-subset] policy note -> results\eda\subset_policy.md
[prepare-subset] targets=['cs.AI', 'cs.CL', 'cs.LG', 'stat.ML'] limit=500
[prepare-subset] DONE scanned=67,170 written=500 invalid=0
All artifacts written:
  data\processed\dev_5k.jsonl  (7156 KB)
  data\processed\submission_sample_under_5mb.jsonl  (676 KB)
  results\eda\subset_policy.md  (1 KB)
  results\data_validation\paperrecord_validation.json  (0 KB)


## 10. Reproducibility & handoff

- **Deterministic subset:** same input + same `limit` → identical corpus.
- **Offline-first:** all demo cells run without the 4 GB file or an API key.
- **Schema-compliant:** every record validated against `PaperRecord` (Section 7).
- **Resumable enrichment:** S2 results disk-cached; safe to interrupt and continue.

### Extended `PaperRecord` fields (backward-compatible — defaults preserve old records)
| Field | Type | Default | Consumer |
| --- | --- | --- | --- |
| `citation_count` | `int` | `0` | Bank — recommendation ranking |
| `influential_citation_count` | `int` | `0` | Bank — ranking signal |
| `references` | `list[str]` | `[]` | Bank / Nadiyah — citation analysis |
| `tldr` | `str` | `""` | Mulkraj — summarisation baseline |
| `s2_enriched` | `bool` | `False` | All — enrichment status flag |

### Downstream handoff
| Member | File | Ready when |
| --- | --- | --- |
| Bank | `dev_5k.jsonl` (unenriched OK) or `dev_5k_enriched.jsonl` | Stage 01 done |
| Mulkraj | `dev_5k_enriched.jsonl` (needs `tldr`) | After S2 key |
| Sidharth | `submission_sample_under_5mb.jsonl` | Stage 01 done |
| Nadiyah | `dev_5k.jsonl` (categories + abstracts) | Stage 01 done |

**Next stages (Yash's track):**
03 EDA & limitations · 04 preprocessing pipeline ·
05 TF-IDF + LogReg / SVM baselines · 06 SPECTER2 classification ·
07 data card · 08 evaluation tables & handoff.
